## Partie 1 – Exploration du dataset

Pour chaque image : nom, classe, format, mode, largeur, hauteur, écart-type des pixels, nombre de canaux, taille (poids en octets). Gestion des fichiers corrompus.

In [2]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

RAW_DIR = Path("../data/raw")
CLASSES = sorted([d.name for d in RAW_DIR.iterdir() if d.is_dir()])
print("Classes détectées :", CLASSES)


Classes détectées : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']


In [4]:
def explore_image(path: Path, classe: str) -> dict:
    """Récupère les métadonnées d'une image. Renvoie un dict avec corrupted=True si l'image est illisible."""
    info = {
        "nom": path.name,
        "classe": classe,
        "chemin": str(path),
        "taille_octets": path.stat().st_size if path.exists() else None,
        "format": None,
        "mode": None,
        "largeur": None,
        "hauteur": None,
        "nb_canaux": None,
        "ecart_type_pixels": None,
        "corrompue": False,
    }
    try:
        with Image.open(path) as img:
            img.verify()  # vérifie l'intégrité du fichier (structure)
        # verify() ferme le fichier -> on le rouvre pour lire les données pixel
        with Image.open(path) as img:
            info["format"] = img.format
            info["mode"] = img.mode
            info["largeur"], info["hauteur"] = img.size
            info["nb_canaux"] = len(img.getbands())
            arr = np.array(img.convert(img.mode))
            info["ecart_type_pixels"] = float(np.std(arr))
    except Exception as e:
        info["corrompue"] = True
        info["erreur"] = str(e)
    return info


In [5]:
records = []
for classe in CLASSES:
    class_dir = RAW_DIR / classe
    for path in sorted(class_dir.iterdir()):
        if path.is_file():
            records.append(explore_image(path, classe))

df = pd.DataFrame(records)
print(f"Nombre total d'images explorées : {len(df)}")
df.head()


Nombre total d'images explorées : 1032


,nom,classe,chemin,taille_octets,format,mode,largeur,hauteur,nb_canaux,ecart_type_pixels,corrompue,erreur
0,cardboard1.jpg,cardboard,../data/raw/cardboard/cardboard1.jpg,17333,JPEG,RGB,512.0,384.0,3.0,40.588529,False,NaN
1,cardboard10.jpg,cardboard,../data/raw/cardboard/cardboard10.jpg,21683,JPEG,RGB,512.0,384.0,3.0,42.571288,False,NaN
2,cardboard100.jpg,cardboard,../data/raw/cardboard/cardboard100.jpg,14884,JPEG,RGB,512.0,384.0,3.0,46.108305,False,NaN
3,cardboard101.jpg,cardboard,../data/raw/cardboard/cardboard101.jpg,14289,JPEG,RGB,512.0,384.0,3.0,72.263996,False,NaN
4,cardboard102.jpg,cardboard,../data/raw/cardboard/cardboard102.jpg,18015,JPEG,RGB,512.0,384.0,3.0,48.388937,False,NaN


In [6]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 1032 entries, 0 to 1031
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   nom                1032 non-null   str    
 1   classe             1032 non-null   str    
 2   chemin             1032 non-null   str    
 3   taille_octets      1032 non-null   int64  
 4   format             1026 non-null   str    
 5   mode               1026 non-null   str    
 6   largeur            1026 non-null   float64
 7   hauteur            1026 non-null   float64
 8   nb_canaux          1026 non-null   float64
 9   ecart_type_pixels  1026 non-null   float64
 10  corrompue          1032 non-null   bool   
 11  erreur             6 non-null      str    
dtypes: bool(1), float64(4), int64(1), str(6)
memory usage: 89.8 KB
